# T09 - Master Pipeline & Demo Dự Đoán Giá Xe Cũ
Dự án: USED CAR PRICE PREDICTION - ADY201m (Group 2)

Notebook này tổng hợp toàn bộ luồng hoạt động từ T01 đến T08 và xây dựng hệ thống demo dự đoán giá xe từ thông số đầu vào thực tế.

## 1. Tổng quan quy trình end-to-end (T01 -> T08)

| Giai đoạn | Nội dung thực hiện | Đầu ra chính |
| :--- | :--- | :--- |
| **T01 - Thu thập dữ liệu** | Thu thập tin đăng từ Chợ Tốt Xe API | `Data/crawler/data/cars.csv` (12.494 dòng) |
| **T02 - Tiền xử lý dữ liệu** | Loại bỏ tin trùng, xử lý missing values, chuẩn hóa ODO và giá (triệu VNĐ) | `Data/data_cleaning/cars_cleaned.csv` |
| **T03 - Phân tích dữ liệu (EDA)** | Phân bố giá theo hãng, khấu hao theo năm sản xuất, tác động của ODO | Biểu đồ phân tích và kết luận thị trường |
| **T04 - Kỹ thuật đặc trưng** | Tách cờ chính chủ, cờ zin, tính tuổi xe, One-Hot Encoding, Scaling trên Train | `preprocessor.joblib`, `data_train.csv`, `data_test.csv` |
| **T05 - Mô hình cơ sở** | Huấn luyện Linear Regression baseline | R² ≈ 0.612 trên tập test |
| **T06 - Mô hình nâng cao** | Huấn luyện Random Forest, XGBoost và Hyperparameter Tuning | `best_model.pkl` (XGBoost Tuned, R² ≈ 0.760) |
| **T07 - Đánh giá mô hình** | So sánh đa chiều (R², MAE, RMSE, MAPE) trên tập test độc lập | Bảng tổng hợp hiệu năng mô hình |
| **T08 - Trực quan hóa độ quan trọng** | Phân tích Feature Importance và Residuals | Top đặc trưng: Hãng xe sang, hộp số sàn, tuổi xe |
| **T09 - Demo hoàn chỉnh** | Tải mô hình và bộ tiền xử lý đã lưu, nhập thông số xe và xuất giá dự đoán | Module `CarPricePredictor` và kịch bản demo |

### Khai báo thư viện và nạp môi trường

In [1]:
import os
import sys
import json
import warnings
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
warnings.filterwarnings('ignore', category=UserWarning)

# Đảm bảo đường dẫn tìm kiếm module hỗ trợ cả khi chạy từ Root hoặc từ Demo
for p in [
    os.path.abspath("Demo"),
    os.path.abspath("."),
    os.path.abspath(".."),
    os.path.abspath(os.path.join("..", "Demo"))
]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    from car_predictor import CarPricePredictor
except ModuleNotFoundError:
    from Demo.car_predictor import CarPricePredictor

## Cơ chế lưu và nạp mô hình (Artifacts Persistence)

Trong quy trình huấn luyện máy học thực tế, việc lưu trữ cả **Mô hình dự đoán** và **Bộ tiền xử lý dữ liệu** là điều kiện tiên quyết để hệ thống có thể hoạt động ở giai đoạn demo hoặc triển khai thực tế.

- `joblib.dump(best_model, path_model)`: Lưu cấu trúc và trọng số mô hình tốt nhất (XGBoost Regressor).
- `joblib.dump(preprocessor, path_prep)`: Lưu toàn bộ tham số chuẩn hóa (StandardScaler) và từ điển mã hóa (OneHotEncoder).
- `feature_names.json`: Lưu danh sách thứ tự 134 cột sau biến đổi.

In [2]:
# Xác định đường dẫn file mô hình và tiền xử lý
MODEL_PATH = os.path.join("..", "Modeling", "T06", "best_model.pkl")
PREPROCESSOR_PATH = os.path.join("..", "Feature Engineering", "data_processed", "preprocessor.joblib")
FEATURE_NAMES_PATH = os.path.join("..", "Feature Engineering", "data_processed", "feature_names.json")

# Fallback nếu notebook chạy từ thư mục gốc
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = os.path.join("Modeling", "T06", "best_model.pkl")
    PREPROCESSOR_PATH = os.path.join("Feature Engineering", "data_processed", "preprocessor.joblib")
    FEATURE_NAMES_PATH = os.path.join("Feature Engineering", "data_processed", "feature_names.json")

print("Đường dẫn mô hình:", MODEL_PATH, "(Tồn tại:", os.path.exists(MODEL_PATH), ")")
print("Đường dẫn tiền xử lý:", PREPROCESSOR_PATH, "(Tồn tại:", os.path.exists(PREPROCESSOR_PATH), ")")
print("Đường dẫn danh sách cột:", FEATURE_NAMES_PATH, "(Tồn tại:", os.path.exists(FEATURE_NAMES_PATH), ")")

# Khởi tạo predictor
predictor = CarPricePredictor(
    model_path=MODEL_PATH,
    preprocessor_path=PREPROCESSOR_PATH,
    feature_names_path=FEATURE_NAMES_PATH,
    current_year=2026
)
print("Hệ thống dự đoán giá đã sẵn sàng.")

Đường dẫn mô hình: ..\Modeling\T06\best_model.pkl (Tồn tại: True )
Đường dẫn tiền xử lý: ..\Feature Engineering\data_processed\preprocessor.joblib (Tồn tại: True )
Đường dẫn danh sách cột: ..\Feature Engineering\data_processed\feature_names.json (Tồn tại: True )
Hệ thống dự đoán giá đã sẵn sàng.


## Demo các kịch bản thực tế theo từng phân khúc xe

Phần này thử nghiệm mô hình trên 4 mẫu xe đại diện cho các phân khúc phổ biến trên thị trường xe cũ Việt Nam:
1. **Sedan phổ thông**: Toyota Vios đời 2022, 30.000 km, máy xăng, số tự động.
2. **SUV hạng sang**: Mercedes-Benz GLC đời 2020, 50.000 km, máy xăng, số tự động.
3. **Bán tải máy dầu**: Ford Ranger đời 2021, 65.000 km, máy dầu, số tự động.
4. **Xe điện đô thị**: VinFast VF3 đời 2024, 6.000 km, máy điện, số tự động.

In [4]:
demo_cases = [
    {
        "ten_xe": "Toyota Vios 2022 (Sedan bình dân)",
        "params": {
            "carbrand_name": "Toyota",
            "carmodel_name": "Vios",
            "mfdate": 2022,
            "odo_km": 30000,
            "gearbox": "Số tự động",
            "fuel": "Xăng",
            "cartype": "Sedan",
            "carorigin": "Lắp ráp trong nước",
            "region_name": "Hà Nội",
            "subject": "Toyota Vios 2022 bản G chính chủ xe gia đình giữ gìn máy zin"
        }
    },
    {
        "ten_xe": "Mercedes-Benz GLC 2020 (SUV hạng sang)",
        "params": {
            "carbrand_name": "Mercedes Benz",
            "carmodel_name": "GLC",
            "mfdate": 2020,
            "odo_km": 50000,
            "gearbox": "Số tự động",
            "fuel": "Xăng",
            "cartype": "SUV / Crossover",
            "carorigin": "Lắp ráp trong nước",
            "region_name": "Hồ Chí Minh",
            "subject": "Mercedes GLC 300 2020 full lịch sử bảo dưỡng hãng sơn zin"
        }
    },
    {
        "ten_xe": "Ford Ranger 2021 (Bán tải máy dầu)",
        "params": {
            "carbrand_name": "Ford",
            "carmodel_name": "Ranger",
            "mfdate": 2021,
            "odo_km": 65000,
            "gearbox": "Số tự động",
            "fuel": "Dầu",
            "cartype": "Bán tải",
            "carorigin": "Nhập khẩu Thái Lan",
            "region_name": "Bình Dương",
            "subject": "Ford Ranger Wildtrak 2021 nhập Thái máy zin một chủ"
        }
    },
    {
        "ten_xe": "VinFast VF3 2024 (Xe điện mini)",
        "params": {
            "carbrand_name": "VinFast",
            "carmodel_name": "VF3",
            "mfdate": 2024,
            "odo_km": 6000,
            "gearbox": "Số tự động",
            "fuel": "Điện",
            "cartype": "Hatchback",
            "carorigin": "Lắp ráp trong nước",
            "region_name": "Hà Nội",
            "subject": "VinFast VF3 2024 chính chủ siêu lướt pin zin"
        }
    }
]

records = []
for case in demo_cases:
    out = predictor.predict(case["params"])
    records.append({
        "Phân khúc thử nghiệm": case["ten_xe"],
        "Năm SX": case["params"]["mfdate"],
        "Số ODO (km)": f"{case['params']['odo_km']:,}",
        "Nhiên liệu": case["params"]["fuel"],
        "Hộp số": case["params"]["gearbox"],
        "Giá dự đoán (Triệu VNĐ)": out["predicted_price_million"],
        "Giá quy đổi (VNĐ)": out["predicted_price_vnd"]
    })

df_demo_res = pd.DataFrame(records)
df_demo_res

,Phân khúc thử nghiệm,Năm SX,Số ODO (km),Nhiên liệu,Hộp số,Giá dự đoán (Triệu VNĐ),Giá quy đổi (VNĐ)
0,Toyota Vios 2022 (Sedan bình dân),2022,"30,000",Xăng,Số tự động,399.9,399.900.000 VND
1,Mercedes-Benz GLC 2020 (SUV hạng sang),2020,"50,000",Xăng,Số tự động,1065.5,1.065.500.000 VND
2,Ford Ranger 2021 (Bán tải máy dầu),2021,"65,000",Dầu,Số tự động,522.2,522.200.000 VND
3,VinFast VF3 2024 (Xe điện mini),2024,"6,000",Điện,Số tự động,199.2,199.200.000 VND


## Ô tương tác: Nhập thông số xe tùy chọn để dự đoán giá

Để thử nghiệm với một chiếc xe bất kỳ, bạn chỉ cần thay đổi các giá trị trong dictionary `thong_so_xe` bên dưới và chạy cell.

In [5]:
# Chỉnh sửa thông số chiếc xe bạn muốn định giá tại đây:
thong_so_xe = {
    "carbrand_name": "Honda",          # Hãng xe: Toyota, Honda, Hyundai, Kia, Mazda, Ford, VinFast...
    "carmodel_name": "City",            # Dòng xe: Vios, City, Accent, CX 5, Ranger, Santa Fe...
    "mfdate": 2021,                     # Năm sản xuất
    "odo_km": 40000,                    # Số km đã đi
    "gearbox": "Số tự động",            # Hộp số: "Số tự động" hoặc "Số sàn"
    "fuel": "Xăng",                     # Nhiên liệu: "Xăng", "Dầu", "Điện", "Hybrid"
    "cartype": "Sedan",                 # Kiểu dáng: "Sedan", "SUV / Crossover", "Hatchback", "Bán tải", "MPV"
    "carorigin": "Lắp ráp trong nước",  # Xuất xứ: "Lắp ráp trong nước", "Nhập khẩu Thái Lan"...
    "region_name": "Hồ Chí Minh",       # Địa phương: "Hà Nội", "Hồ Chí Minh", "Đà Nẵng"...
    "flag_chinh_chu": 1,                # 1: Chính chủ / 1 chủ từ đầu, 0: Không rõ
    "flag_zin": 1,                      # 1: Xe cam kết zin nguyên bản, 0: Không rõ
    "flag_bao_duong_hang": 1            # 1: Có lịch sử hãng đầy đủ, 0: Không rõ
}

ket_qua = predictor.predict(thong_so_xe)

print("=" * 55)
print("KET QUA DINH GIA XE CU")
print("=" * 55)
print(f"Xe:           {thong_so_xe['carbrand_name']} {thong_so_xe['carmodel_name']} ({thong_so_xe['mfdate']})")
print(f"ODO:          {thong_so_xe['odo_km']:,} km")
print(f"Hop so:       {thong_so_xe['gearbox']} | Nhien lieu: {thong_so_xe['fuel']}")
print(f"Kieu dang:    {thong_so_xe['cartype']} | Xuat xu: {thong_so_xe['carorigin']}")
print(f"Khu vuc:      {thong_so_xe['region_name']}")
print(f"Gia du doan:  {ket_qua['predicted_price_million']} trieu VND ({ket_qua['predicted_price_vnd']})")
print("=" * 55)

KET QUA DINH GIA XE CU
Xe:           Honda City (2021)
ODO:          40,000 km
Hop so:       Số tự động | Nhien lieu: Xăng
Kieu dang:    Sedan | Xuat xu: Lắp ráp trong nước
Khu vuc:      Hồ Chí Minh
Gia du doan:  400.6 trieu VND (400.600.000 VND)


## 7. Giao diện trực quan tùy chọn (Interactive UI Widgets)

Nếu chạy trong Jupyter Notebook hoặc VS Code, bạn có thể tương tác trực tiếp với giao diện chọn thả bên dưới.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    brand_w = widgets.Dropdown(options=sorted(list(predictor.known_brands)), value="Toyota", description="Hãng xe:")
    model_w = widgets.Dropdown(options=sorted(list(predictor.known_models)), value="Vios", description="Dòng xe:")
    year_w = widgets.IntSlider(value=2021, min=2010, max=2025, step=1, description="Năm SX:")
    odo_w = widgets.IntSlider(value=40000, min=0, max=300000, step=5000, description="ODO (km):")
    gear_w = widgets.Dropdown(options=["Số tự động", "Số sàn"], value="Số tự động", description="Hộp số:")
    fuel_w = widgets.Dropdown(options=["Xăng", "Dầu", "Điện", "Hybrid"], value="Xăng", description="Nhiên liệu:")
    cartype_w = widgets.Dropdown(options=["Sedan", "SUV / Crossover", "Hatchback", "Bán tải", "MPV"], value="Sedan", description="Kiểu dáng:")
    origin_w = widgets.Dropdown(options=["Lắp ráp trong nước", "Nhập khẩu Thái Lan", "Nhập khẩu Indonesia", "Nhập khẩu Hàn Quốc", "Nhập khẩu Nhật Bản"], value="Lắp ráp trong nước", description="Xuất xứ:")
    region_w = widgets.Dropdown(options=sorted(list(predictor.known_regions)), value="Hà Nội", description="Khu vực:")
    chinhchu_w = widgets.Checkbox(value=True, description="Chính chủ / 1 chủ")
    zin_w = widgets.Checkbox(value=True, description="Cam kết nguyên zin")
    btn_predict = widgets.Button(description="Dự đoán giá xe", button_style="primary")
    out_box = widgets.Output()

    def on_click_predict(b):
        with out_box:
            clear_output()
            user_car = {
                "carbrand_name": brand_w.value,
                "carmodel_name": model_w.value,
                "mfdate": year_w.value,
                "odo_km": odo_w.value,
                "gearbox": gear_w.value,
                "fuel": fuel_w.value,
                "cartype": cartype_w.value,
                "carorigin": origin_w.value,
                "region_name": region_w.value,
                "flag_chinh_chu": int(chinhchu_w.value),
                "flag_zin": int(zin_w.value),
                "flag_bao_duong_hang": 0
            }
            res = predictor.predict(user_car)
            print(f"Giá dự đoán: {res['predicted_price_million']} triệu VNĐ ({res['predicted_price_vnd']})")

    btn_predict.on_click(on_click_predict)
    ui = widgets.VBox([
        widgets.HBox([brand_w, model_w]),
        widgets.HBox([year_w, odo_w]),
        widgets.HBox([gear_w, fuel_w]),
        widgets.HBox([cartype_w, origin_w]),
        widgets.HBox([region_w]),
        widgets.HBox([chinhchu_w, zin_w]),
        btn_predict,
        out_box
    ])
    display(ui)
except ImportError:
    print("ipywidgets chưa được cài đặt. Có thể sử dụng ô nhập từ điển tại mục 6.")